# Лабораторна робота 4 — LASSO-регресія через координатний спуск

**Набір даних:** `kc_house_data.csv`  
**Обмеження:** scikit-learn LASSO **не дозволено** для базових завдань.

## Налаштування

In [14]:
from google.colab import drive
import sys
drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/MTAD_Labs/Lab4')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import sys
!{sys.executable} -m pip install numpy pandas matplotlib --quiet


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

%matplotlib inline


## Теоретичне підґрунтя

Для нормалізованих ознак значення ρᵢ для координатного спуску LASSO:
```
ρᵢ = dot( featureᵢ,  output − (Xw − wᵢ·featureᵢ) )
```
Правило м'якого порогування для wᵢ (i ≥ 1):
```
wᵢ = ρᵢ + λ/2    якщо ρᵢ < −λ/2
wᵢ = 0           якщо −λ/2 ≤ ρᵢ ≤ λ/2
wᵢ = ρᵢ − λ/2    якщо ρᵢ >  λ/2
```
Вільний член (i = 0) **не** регуляризується: w₀ = ρ₀.

## Допоміжні функції — `get_numpy_data` і `predict_output` (повторне використання з Лаб. 3)

Наведено для зручності. Якщо ви виконали Лабораторну роботу 3, можете підставити власну реалізацію.

In [4]:
def get_numpy_data(dataframe, features, output):
    """Будує матрицю ознак зі стовпцем вільного члена та вектор виходу."""
    dataframe = dataframe.copy()
    dataframe['constant'] = 1.0
    feature_matrix = dataframe[['constant'] + list(features)].to_numpy(dtype=float)
    output_array   = dataframe[output].to_numpy(dtype=float)
    return feature_matrix, output_array


In [5]:
def predict_output(feature_matrix, weights):
    """Передбачення через скалярний добуток."""
    return np.dot(feature_matrix, weights)


---
## Завдання 1 — Нормалізація ознак

Реалізуйте `normalize_features(feature_matrix)`, яка ділить кожен стовпець на його норму L2 та повертає `(normalised_features, norms)`.

In [6]:
def normalize_features(feature_matrix):
    """
    Ділить кожен стовпець на його норму L2.

    Повертає
    -------
    normalised : np.ndarray, та сама форма що й feature_matrix
    norms      : np.ndarray, shape (n_cols,)
    """
    norms      = np.linalg.norm(feature_matrix, axis=0)

    normalised = feature_matrix / norms

    return normalised, norms


### Перевірка — кожен стовпець `normalised` має дорівнювати [0.6, 0.8]; norms = [5, 10, 15]

In [7]:
test_matrix = np.array([[3., 6., 9.], [4., 8., 12.]])
normalised, norms = normalize_features(test_matrix)
print('Нормалізована матриця:\n', normalised)
print('Норми:', norms)


Нормалізована матриця:
 [[0.6 0.6 0.6]
 [0.8 0.8 0.8]]
Норми: [ 5. 10. 15.]


---
## Завдання 2 — Один крок координатного спуску

Реалізуйте `lasso_coordinate_descent_step(i, feature_matrix, output, weights, l1_penalty)`. Функція має:
1. Обчислити поточне передбачення.
2. Обчислити ρᵢ.
3. Якщо i = 0: wᵢ = ρᵢ (без штрафу). Якщо i ≥ 1: застосувати правило м'якого порогування.
4. Повернути нове значення ваги.

In [8]:
def lasso_coordinate_descent_step(i, feature_matrix, output, weights, l1_penalty):
    """
    Виконує один крок координатного спуску LASSO для ваги з індексом i.

    Параметри
    ----------
    i             : int    індекс ваги для оновлення
    feature_matrix: array  нормалізована, shape (n, d)
    output        : array  shape (n,)
    weights       : array  поточні ваги, shape (d,)
    l1_penalty    : float  λ

    Повертає
    -------
    new_weight_i  : float
    """
    prediction = predict_output(feature_matrix, weights)

    # Compute rho_i
    rho_i = np.dot(feature_matrix[:, i], (output - prediction + weights[i] * feature_matrix[:, i]))

    # Apply soft-threshold (intercept is unregularised)
    if i == 0:   # intercept — no penalty
        new_weight_i = rho_i
    elif rho_i < -l1_penalty / 2:
        new_weight_i = rho_i + (l1_penalty / 2)
    elif rho_i > l1_penalty / 2:
        new_weight_i = rho_i - (l1_penalty / 2)
    else:
        new_weight_i = 0.0

    return new_weight_i


### Перевірка — результат має бути ≈ 0.4256

In [9]:
from math import sqrt
result = lasso_coordinate_descent_step(
    i=1,
    feature_matrix=np.array([[3/sqrt(13), 1/sqrt(10)],
                             [2/sqrt(13), 3/sqrt(10)]]),
    output=np.array([1., 1.]),
    weights=np.array([1., 4.]),
    l1_penalty=0.1
)
print(f'Результат кроку: {result:.4f}  (очікується ≈ 0.4256)')


Результат кроку: 0.4256  (очікується ≈ 0.4256)


---
## Завдання 3 — Циклічний координатний спуск

Реалізуйте `lasso_cyclical_coordinate_descent(feature_matrix, output, initial_weights, l1_penalty, tolerance)`. У кожному циклі оновлюйте ваги 0, 1, …, d−1 по черзі, використовуючи оновлену вагу одразу. Зупиняйтесь, коли максимальна абсолютна зміна ваги за цикл < `tolerance`.

In [10]:
def lasso_cyclical_coordinate_descent(feature_matrix, output,
                                       initial_weights, l1_penalty, tolerance):
    """
    Запускає циклічний координатний спуск до збіжності.

    Повертає
    -------
    weights : np.ndarray — навчені ваги (на нормалізованих ознаках)
    """
    weights   = np.array(initial_weights, dtype=float)
    converged = False

    while not converged:
        max_change = 0.0
        for i in range(len(weights)):
            old_weight_i = weights[i]
            # Update weight i
            weights[i] = lasso_coordinate_descent_step(i, feature_matrix, output, weights, l1_penalty)

            # Track the largest change this cycle
            max_change = max(max_change, abs(weights[i] - old_weight_i))

        if max_change < tolerance:
            converged = True

    return weights


---
## Завдання 4 — Відбір ознак

Побудуйте нормалізовану матрицю ознак з `kc_house_data.csv`, використовуючи `['sqft_living', 'bedrooms']`. Запустіть координатний спуск з:
```
initial_weights = [0., 0., 0.]
tolerance       = 1.0
```
— `l1_penalty = 1e7`: вкажіть навчені ваги, RSS та які ознаки мають ненульові ваги.  
— `l1_penalty = 1e8`: повторіть. Що змінилося?

In [11]:
sales = pd.read_csv('/content/drive/MyDrive/MTAD_Labs/Lab4/kc_house_data.csv')

# Побудуйте та нормалізуйте матрицю ознак
feature_matrix, output = get_numpy_data(sales, ['sqft_living', 'bedrooms'], 'price')
normalised_features, feature_norms = normalize_features(feature_matrix)


In [12]:
# Запуск з l1_penalty = 1e7
weights_1e7 = lasso_cyclical_coordinate_descent(
    normalised_features, output,
    initial_weights=np.zeros(3),
    l1_penalty=1e7,
    tolerance=1.0
)
print('Ваги (λ=1e7):', weights_1e7)
print('Non-zero features:', ['intercept', 'sqft_living', 'bedrooms'][
    :len([w for w in weights_1e7 if w != 0])  # rough indicator
])

# RSS on the normalised dataset
preds_1e7 = predict_output(normalised_features, weights_1e7)
rss_1e7   = np.sum((preds_1e7 - output) ** 2)
print(f'RSS (λ=1e7): {rss_1e7:.3e}')


Ваги (λ=1e7): [21624997.95951913 63157247.20788954        0.        ]
Non-zero features: ['intercept', 'sqft_living']
RSS (λ=1e7): 1.630e+15


In [13]:
# Запуск з l1_penalty = 1e8
weights_1e8 = lasso_cyclical_coordinate_descent(
    normalised_features, output,
    initial_weights=np.zeros(3),
    l1_penalty=1e8,
    tolerance=1.0
)
print('Ваги (λ=1e8):', weights_1e8)


Ваги (λ=1e8): [79400304.63764462        0.                0.        ]


**Спостереження:** При штрафі λ = $10^7$ вага ознаки bedrooms стала рівною 0, тоді як sqft_living залишилася в моделі. Це свідчить про те, що житлова площа є набагато вагомішим фактором для прогнозу ціни, ніж кількість спалень.

Зі збільшенням λ до $10^8$ модель стала максимально простою (розрідженою) — усі ваги, крім вільного члена (intercept), занулилися. Штраф виявився настільки великим, що моделі «вигідніше» ігнорувати всі ознаки.

Чим більший коефіцієнт L1, тим менше ознак залишається в моделі. Це підтверджує, що LASSO-регресія ефективно працює як автоматичний інструмент для вибору найважливіших даних.